# `langchain_azure_ai` smoke test

Verifies that the package imports and its primary API surface (chat models, embeddings, vector stores, retrievers, callbacks/tracers) is reachable. No network calls — we only instantiate or inspect classes, we never invoke a remote Azure endpoint.

In [ ]:
# Example 1: chat_models submodule exposes AzureAIChatCompletionsModel.
from langchain_azure_ai import chat_models

ChatCls = chat_models.AzureAIChatCompletionsModel
print(f"class: {ChatCls.__module__}.{ChatCls.__name__}")
print(f"mro head: {[c.__name__ for c in ChatCls.__mro__[:5]]}")

# It's a pydantic-based BaseChatModel; field names should include credential + endpoint.
fields = set(getattr(ChatCls, 'model_fields', {}).keys())
print(f"has 'endpoint' field: {'endpoint' in fields}")
print(f"has 'credential' field: {'credential' in fields}")
assert callable(ChatCls)

In [ ]:
# Example 2: embeddings submodule exposes AzureAIOpenAIApiEmbeddingsModel.
# (AzureAIEmbeddingsModel was replaced by AzureAIOpenAIApiEmbeddingsModel in
# langchain-azure-ai 1.x; AzureOpenAIEmbeddings is also re-exported here.)
from langchain_azure_ai import embeddings

EmbCls = embeddings.AzureAIOpenAIApiEmbeddingsModel
print(f"class: {EmbCls.__module__}.{EmbCls.__name__}")
print(f"mro head: {[c.__name__ for c in EmbCls.__mro__[:5]]}")

# embed_documents / embed_query should be defined methods.
assert hasattr(EmbCls, 'embed_documents')
assert hasattr(EmbCls, 'embed_query')
assert hasattr(embeddings, 'AzureOpenAIEmbeddings')
print("embed_documents + embed_query methods present")

In [ ]:
# Example 3: vectorstores.azuresearch surface — AzureSearch + AzureSearchVectorStoreRetriever.
from langchain_azure_ai.vectorstores import azuresearch

names = [n for n in dir(azuresearch) if not n.startswith('_')]
print(f"public symbols (head): {sorted(names)[:12]}")

VS = azuresearch.AzureSearch
Retriever = azuresearch.AzureSearchVectorStoreRetriever
print(f"AzureSearch base: {VS.__bases__[0].__name__}")
print(f"Retriever base: {Retriever.__bases__[0].__name__}")
assert hasattr(VS, 'add_texts')
assert hasattr(VS, 'similarity_search')

In [ ]:
# Example 4: retrievers.azure_ai_search exposes AzureAISearchRetriever.
from langchain_azure_ai import retrievers

RetrieverCls = retrievers.AzureAISearchRetriever
print(f"class: {RetrieverCls.__module__}.{RetrieverCls.__name__}")
print(f"mro head: {[c.__name__ for c in RetrieverCls.__mro__[:5]]}")

# BaseRetriever subclasses implement _get_relevant_documents.
assert hasattr(RetrieverCls, '_get_relevant_documents') or hasattr(RetrieverCls, 'get_relevant_documents')
print("retriever interface present")

In [ ]:
# Example 5: tracer surface — AzureAIOpenTelemetryTracer now lives in
# callbacks.tracers.inference_tracing (the tracers package __init__ is lazy in 1.x,
# so dir() on it shows no classes — import the concrete submodule directly).
from langchain_azure_ai.callbacks.tracers import auto_instrument
from langchain_azure_ai.callbacks.tracers.inference_tracing import AzureAIOpenTelemetryTracer

print(f"tracer class: {AzureAIOpenTelemetryTracer.__module__}.{AzureAIOpenTelemetryTracer.__name__}")

helpers = [n for n in dir(auto_instrument) if 'auto_tracing' in n]
print(f"auto-instrument helpers: {helpers}")
assert helpers, "expected enable/disable auto-tracing helpers in auto_instrument"
print("OK langchain_azure_ai primary API reachable")